In [ ]:
import pandas as pd
sales=pd.read_csv(r"C:\Users\new\Downloads\sell_prices.csv\sell_prices.csv")
sales
train=pd.read_csv(r"C:\Users\new\Downloads\sales_train_evaluation.csv\sales_train_evaluation.csv")
train
valid=pd.read_csv(r"C:\Users\new\Downloads\sales_train_validation.csv")
valid
humid=pd.read_csv(r"C:\Users\new\Downloads\humidity.csv\humidity.csv")
humid
pressure=pd.read_csv(r"C:\Users\new\Downloads\pressure.csv")
pressure
temp=pd.read_csv(r"C:\Users\new\Downloads\temperature.csv")
temp
weather=pd.read_csv(r"C:\Users\new\Downloads\weather_description.csv")
weather
wind=pd.read_csv(r"C:\Users\new\Downloads\wind_direction.csv")
wind
speed=pd.read_csv(r"C:\Users\new\Downloads\wind_speed.csv\wind_speed.csv")
speed
city=pd.read_csv(r"C:\Users\new\Downloads\city_attributes.csv")
city



import pandas as pd
calendar=pd.read_csv(r"C:\Users\new\Downloads\calendar.csv")
calendar.head()
calendar.info()
calendar.describe()
calendar.isnull().sum()             #checking missing values
cols = ['event_name_1','event_type_1','event_name_2','event_type_2']
calendar[cols] = calendar[cols].fillna('None')
calendar[cols]                     #filled with NaN

calendar["date"]=pd.to_datetime(calendar["date"])
calendar["date"]   #convert date to date tym

calendar.duplicated().sum()  #checked duplicates

calendar.info()
calendar.head(15)
calendar=calendar.rename(columns={
    'snap_CA':'benefit_day_region_1',
    'snap_TX':'benefit_day_region_2',    
    'snap_WI':'benefit_day_region_3'
})
calendar

calendar=calendar.drop(["event_name_2","event_type_2"],axis=1)
calendar

calendar=calendar.rename(columns={
    "event_name_1":"event_name",
    "event_type_1":"event_type"
    })
calendar


import pandas as pd
valid=pd.read_csv(r"C:\Users\new\Downloads\sales_train_validation.csv",nrows=5000)
valid
id_cols = ["id","item_id","dept_id","cat_id","store_id","state_id"]
id_cols
#converting wide format to long format
valid_long = pd.melt(
    valid,
    id_vars=id_cols,
    var_name='d',
    value_name='sales'
)
valid_long.head(20)

#checking null
valid_long.isnull()

#checking shape
valid_long.shape

#merging calendar and valid 
calendar=pd.read_csv(r"C:\Users\new\Downloads\calendar.csv")
calendar.head()
#calendar.info()
#calendar.describe()
calendar.isnull().sum()             #checking missing values
cols = ['event_name_1','event_type_1','event_name_2','event_type_2']
calendar[cols] = calendar[cols].fillna('No_Event')
calendar[cols]                     #filled with NaN

calendar["date"]=pd.to_datetime(calendar["date"])
calendar["date"]   #convert date to date tym

calendar.duplicated().sum()  #checked duplicates

#calendar.info()
#calendar.head(15)
calendar=calendar.rename(columns={
    'snap_CA':'benefit_day_region_1',
    'snap_TX':'benefit_day_region_2',    
    'snap_WI':'benefit_day_region_3'
})
# calendar

calendar=calendar.drop(["event_name_2","event_type_2"],axis=1)
calendar

calendar=calendar.rename(columns={
    "event_name_1":"event_name",
    "event_type_1":"event_type"
    })
calendar

#actual merging
valid_calendar= valid_long.merge(calendar,on="d",how="left")
valid_calendar.head(20)
valid_calendar["date"]=pd.to_datetime(valid_calendar["date"])
#valid_calendar.info()
valid_calendar.shape

#merging prices
price =pd.read_csv(r"C:\Users\new\Downloads\sell_prices.csv\sell_prices.csv")
price_calendar_valid = valid_calendar.merge(price,on=["store_id","item_id","wm_yr_wk"],how="left")
price_calendar_valid.head()

price_calendar_valid["sell_price"] = price_calendar_valid["sell_price"].ffill().bfill()
price_calendar_valid

#checking entire remaining null
price_calendar_valid.isnull().sum()

#reducing memory
cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name",
    "event_type"
]
for col in cols:
    price_calendar_valid[col] = price_calendar_valid[col].astype("category")
# print("conversion completed")

#price_calendar_valid.info()
type(price_calendar_valid)

#feature engineering
price_calendar_valid["day_of_week"] = price_calendar_valid["date"].dt.dayofweek
price_calendar_valid["week_of_year"] = price_calendar_valid["date"].dt.isocalendar().week
price_calendar_valid["month"] = price_calendar_valid["date"].dt.month
price_calendar_valid["year"] = price_calendar_valid["date"].dt.year

price_calendar_valid.columns
price_calendar_valid.head()

# create lag features

price_calendar_valid["lag_7"] = price_calendar_valid.groupby("item_id")["sales"].transform(lambda x: x.shift(7))
price_calendar_valid["lag_14"] = price_calendar_valid.groupby("item_id")["sales"].transform(lambda x: x.shift(14))
price_calendar_valid["lag_21"] = price_calendar_valid.groupby("item_id")["sales"].transform(lambda x: x.shift(21))
price_calendar_valid["lag_28"] = price_calendar_valid.groupby("item_id")["sales"].transform(lambda x: x.shift(28))
price_calendar_valid[["sales","lag_7","lag_14","lag_21","lag_28"]].head(20)

price_calendar_valid=price_calendar_valid.dropna()
#price_calendar_valid.isnull().sum()

#rolling average features
price_calendar_valid["rolling_mean_7"] = price_calendar_valid.groupby("item_id")["sales"].transform(lambda x: x.rolling(7).mean())
price_calendar_valid["rolling_mean_14"] = price_calendar_valid.groupby("item_id")["sales"].transform(lambda x: x.rolling(14).mean())
price_calendar_valid["rolling_mean_28"] = price_calendar_valid.groupby("item_id")["sales"].transform(lambda x: x.rolling(28).mean())

price_calendar_valid=price_calendar_valid.dropna()

price_calendar_valid.head()


#encode categorical columns 
cols = ["item_id","dept_id","cat_id","store_id","state_id"]

for col in cols:
    price_calendar_valid[col] = price_calendar_valid[col].astype("category").cat.codes

#price_calendar_valid.head()
#price_calendar_valid.info()
#price_calendar_valid.shape

#Extra business features
##a) Sales growth
price_calendar_valid["sales_growth"] = (
    price_calendar_valid["sales"] -
    price_calendar_valid["lag_7"]
)
price_calendar_valid[["sales","lag_7","sales_growth"]].head()
#price_calendar_valid.info()
price_calendar_valid.head()


##b) B — Price Change
price_calendar_valid["price_change"] = (
    price_calendar_valid["sell_price"] -
    price_calendar_valid.groupby("item_id")["sell_price"].shift(1)
)
price_calendar_valid[["item_id","sell_price","price_change"]].head()


##c) C — Weekend Flag
price_calendar_valid["is_weekend"] = (
    price_calendar_valid["day_of_week"] >= 5
).astype(int)
price_calendar_valid[["date","day_of_week","is_weekend"]].head()


##d)D — Revenue
price_calendar_valid["revenue"] = (
    price_calendar_valid["sales"] *
    price_calendar_valid["sell_price"]
)
price_calendar_valid[["sales","sell_price","revenue"]].head()

##e) E — Demand Category
import pandas as pd
price_calendar_valid["demand_level"] = pd.qcut(
    price_calendar_valid["sales"],
    q=3,
    duplicates="drop"
)
price_calendar_valid[["sales","demand_level"]].head()

#merging weather data set
import pandas as pd
import numpy as np
weather=pd.read_csv(r"C:\Users\new\Downloads\weather_description.csv")
humid=pd.read_csv(r"C:\Users\new\Downloads\humidity.csv\humidity.csv")
temp=pd.read_csv(r"C:\Users\new\Downloads\temperature.csv")
speed=pd.read_csv(r"C:\Users\new\Downloads\wind_speed.csv\wind_speed.csv")
#convert datetime to date:Weather is hourly, sales is daily.
for df in [weather, humid, temp, speed]:
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["date"] = df["datetime"].dt.date

#Select One City:Use San Francisco
weather = weather[["date","San Francisco"]]
humid = humid[["date","San Francisco"]]
temp = temp[["date","San Francisco"]]
speed = speed[["date","San Francisco"]]

#Rename Columns
weather = weather.rename(columns={"San Francisco":"weather_desc"})
humid = humid.rename(columns={"San Francisco":"humidity"})
temp = temp.rename(columns={"San Francisco":"temperature"})
speed = speed.rename(columns={"San Francisco":"wind_speed"})

#Convert Hourly → Daily
temp = temp.groupby("date").mean().reset_index()
humid = humid.groupby("date").mean().reset_index()
speed = speed.groupby("date").mean().reset_index()
#Weather description → most frequent.
weather = weather.groupby("date")["weather_desc"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).reset_index()

# Merge Weather Variables
weather_final = temp.merge(humid,on="date",how="left")
weather_final = weather_final.merge(speed,on="date",how="left")
weather_final = weather_final.merge(weather,on="date",how="left")
weather_final.head()

#Convert Date Format In Main Dataset
price_calendar_valid["date"] = pd.to_datetime(price_calendar_valid["date"]).dt.date

#Merge Weather With Main Dataset
price_calendar_valid = price_calendar_valid.merge(
    weather_final,
    on="date",
    how="left"
)
#Handle Missing Values:Forward fill weather values.
price_calendar_valid["temperature"]=price_calendar_valid["temperature"].ffill()
price_calendar_valid["humidity"]=price_calendar_valid["humidity"].ffill()
price_calendar_valid["wind_speed"]=price_calendar_valid["wind_speed"].ffill()

##Feature Engineering (Weather)

#Create Weather Type Feature
def weather_group(x):

    x = str(x).lower()

    if "rain" in x:
        return "rain"

    elif "cloud" in x:
        return "cloudy"

    elif "clear" in x:
        return "clear"

    elif "mist" in x or "fog" in x:
        return "fog"

    elif "snow" in x:
        return "snow"

    else:
        return "other"
price_calendar_valid["weather_type"] = price_calendar_valid["weather_desc"].apply(weather_group)
#encode
price_calendar_valid["weather_type"] = (
    price_calendar_valid["weather_type"]
    .astype("category")
    .cat.codes
)

#Create Weather Severity Feature:This helps ML understand bad vs good weather impact.
def weather_severity(x):

    x = str(x).lower()

    if "rain" in x or "storm" in x:
        return "bad"

    elif "cloud" in x or "mist" in x:
        return "moderate"

    elif "clear" in x:
        return "good"

    else:
        return "moderate"
price_calendar_valid["weather_severity"] = price_calendar_valid["weather_desc"].apply(weather_severity)
# Encode:
price_calendar_valid["weather_severity"] = (
    price_calendar_valid["weather_severity"]
    .astype("category")
    .cat.codes
)
price_calendar_valid.drop(columns=["weather_desc"], inplace=True)


#Temperature Category
price_calendar_valid["temp_level"] = pd.cut(
    price_calendar_valid["temperature"],
    bins=3,
    labels=["cold","mild","hot"]
)
price_calendar_valid["temp_level"] = price_calendar_valid["temp_level"].astype("category").cat.codes

#Humidity Category
price_calendar_valid["humidity_level"] = pd.cut(
    price_calendar_valid["humidity"],
    bins=3,
    labels=["low","medium","high"]
)

price_calendar_valid["humidity_level"] = price_calendar_valid["humidity_level"].astype("category").cat.codes

#Wind Category
price_calendar_valid["wind_level"] = pd.cut(
    price_calendar_valid["wind_speed"],
    bins=3,
    labels=["calm","moderate","strong"]
)

price_calendar_valid["wind_level"] = price_calendar_valid["wind_level"].astype("category").cat.codes


#Final Dataset Check
price_calendar_valid.head()

# Final Dataset Check

print("Start date:", price_calendar_valid["date"].min())
print("End date:", price_calendar_valid["date"].max())

print("Years used:", price_calendar_valid["year"].unique())

print("Total years:", price_calendar_valid["year"].nunique())

price_calendar_valid.groupby("year")["date"].nunique()

sample_items = (
    price_calendar_valid
    .groupby("cat_id")["item_id"]
    .unique()
    .apply(lambda x: x[:10])   # 10 items from each category
)

sample_items = [item for sublist in sample_items for item in sublist]

eda_data = price_calendar_valid[
    price_calendar_valid["item_id"].isin(sample_items)
]

#creating demand volatility/rolling std deviation (7 days)
price_calendar_valid["rolling_std_7"] = (
    price_calendar_valid.groupby("item_id")["sales"]
    .transform(lambda x: x.rolling(7).std())
)
#sales momentum
price_calendar_valid["sales_momentum"] = (
    price_calendar_valid["sales"] /
    price_calendar_valid["lag_7"]
)

price_calendar_valid.dropna()
price_calendar_valid.head(10)


#Exploratory Data Analysis (EDA)
#step 0:First understand the datset
# price_calendar_valid.shape
# price_calendar_valid.columns
# price_calendar_valid.head()

#fixing nan values
#i)fixing weather columns
price_calendar_valid[['temperature','humidity','wind_speed']] = \
price_calendar_valid[['temperature','humidity','wind_speed']].ffill().bfill()

#Filling price change values
price_calendar_valid['price_change'].fillna(0, inplace=True)

#Filling rolling_std_7 nan values
price_calendar_valid['rolling_std_7'].fillna(0, inplace=True)

#Filling sales_momemtum nan values
price_calendar_valid['sales_momentum'].fillna(0, inplace=True)

price_calendar_valid.isnull().sum()
#price_calendar_valid.info()
#price_calendar_valid.describe()

#Step1: Total sales over time(Trend analysis)
#i) Create dataset for EDA(last 4 years only)
# create EDA dataset (last 4 years only)
eda_data = price_calendar_valid[
    price_calendar_valid["year"] >= price_calendar_valid["year"].max() - 3
]
eda_data.shape
eda_data["year"].unique()


#Step 2: Daily sales trend
import matplotlib.pyplot as plt

daily_sales = eda_data.groupby("date")["sales"].sum()

plt.figure(figsize=(15,6))
daily_sales.plot()

plt.title("Total Daily Sales Trend (Last 4 Years)")
plt.xlabel("Date")
plt.ylabel("Total Sales")

plt.show()

#Step 3: Weekly pattern 
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

sns.boxplot(
    x="day_of_week",
    y="sales",
    data=eda_data
)

plt.title("Sales Distribution by Day of Week")
plt.xlabel("Day of Week (0=Monday, 6=Sunday)")
plt.ylabel("Sales")

plt.show()

#step 4: monthly pattern
monthly_sales = eda_data.groupby("month")["sales"].mean()

import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

monthly_sales.plot(kind="bar")

plt.title("Average Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Sales")

plt.show()

#Step 5: Yearly pattern
year_sales = eda_data.groupby("year")["sales"].sum()

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

year_sales.plot(kind="bar")

plt.title("Total Sales by Year")
plt.xlabel("Year")
plt.ylabel("Total Sales")

plt.show()

#STEP 6: Category demand analysis

eda_data["category_name"] = eda_data["cat_id"].map({
0: "HOBBIES",
1: "HOUSEHOLD",
2: "FOODS"
})

cat_sales = eda_data.groupby("category_name")["sales"].sum()

plt.figure(figsize=(8,5))
cat_sales.plot(kind="bar")

plt.title("Total Sales by Product Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")

plt.show()
#price_calendar_valid["cat_id"].value_counts()

#Step 7: Store demand analysis
# create readable store names
price_calendar_valid["store_name"] = price_calendar_valid["store_id"].map({
    0: "CA_1",
    1: "CA_2"
})
price_calendar_valid[["store_id","store_name"]].head()

#Filter only california stores
ca_data = price_calendar_valid[
    price_calendar_valid["store_name"].isin(["CA_1","CA_2"])
]
ca_data["store_name"].unique()

#Calculate total sales per store
store_sales = ca_data.groupby("store_name")["sales"].sum()

print(store_sales)

#Plotting
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

store_sales.plot(kind="bar")

plt.title("Sales Comparison Between California Stores")
plt.xlabel("Store")
plt.ylabel("Total Sales")

plt.show()


#Step 8: Department analysis
price_calendar_valid["dept_id"].unique()
dept_map = {
0: "FOODS_1",
1: "FOODS_2",
2: "FOODS_3",
3: "HOBBIES_1",
4: "HOBBIES_2",
5: "HOUSEHOLD_1",
6: "HOUSEHOLD_2"
}

price_calendar_valid["dept_name"] = price_calendar_valid["dept_id"].map(dept_map)


price_calendar_valid[["dept_id","dept_name"]].head()


dept_sales=price_calendar_valid.groupby("dept_name")["sales"].sum()
print(dept_sales)


import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

dept_sales.sort_values().plot(kind="barh")

plt.title("Department Demand Analysis")
plt.xlabel("Total Sales")
plt.ylabel("Department")

plt.show()

#Step 9:Top Product (Item) Demand Analysis
item_map = dict(zip(valid["item_id"].astype("category").cat.codes, valid["item_id"]))
price_calendar_valid["item_name"] = price_calendar_valid["item_id"].map(item_map)
price_calendar_valid[["item_id","item_name"]].head()

top_items = price_calendar_valid.groupby("item_name")["sales"].sum().sort_values(ascending=False).head(10)

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

top_items.sort_values().plot(kind="barh")

plt.title("Top 10 Selling Products")
plt.xlabel("Total Sales")
plt.ylabel("Item Name")

plt.show()
price_calendar_valid.groupby("cat_id")["sales"].sum()

#Price impact analysis
price_calendar_valid["price_bin"] = pd.cut(
    price_calendar_valid["sell_price"],
    bins=10
)

price_trend = price_calendar_valid.groupby("price_bin")["sales"].mean()

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

price_trend.plot(marker='o')

plt.title("Average Sales vs Price Range")
plt.xlabel("Price Range")
plt.ylabel("Average Sales")

plt.xticks(rotation=45)

plt.show()

#weather impact analysis
#i)Temperature vs Sales
# take sample to avoid memory issue
# take sample to avoid memory issue
eda_sample = price_calendar_valid.sample(n=50000, random_state=42)

# map numbers to names
temp_map = {
    0: "Cold",
    1: "Mild",
    2: "Hot"
}

eda_sample["temp_name"] = eda_sample["temp_level"].map(temp_map)

# groupby using readable names
temp_sales = eda_sample.groupby("temp_name")["sales"].mean()

import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

temp_sales.plot(kind="bar")

plt.title("Average Sales by Temperature Level")
plt.xlabel("Temperature")
plt.ylabel("Average Sales")

plt.show()

#ii)weather type analysis
# take sample (important)
eda_sample = price_calendar_valid.sample(n=50000, random_state=42)

# map weather names
weather_map = {
    0: "Clear",
    1: "Cloudy",
    2: "Rain",
    3: "Fog"
}

eda_sample["weather_name"] = eda_sample["weather_type"].map(weather_map)

# groupby
weather_sales = eda_sample.groupby("weather_name")["sales"].mean()

import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

weather_sales.plot(kind="bar")

plt.title("Average Sales by Weather Type")
plt.xlabel("Weather Type")
plt.ylabel("Average Sales")

plt.show()
price_calendar_valid.columns

#iii)humidity impact
eda_sample = price_calendar_valid.sample(n=50000, random_state=42)

humidity_sales = eda_sample.groupby("humidity_level")["sales"].mean()

humidity_map = {
    0: "Low",
    1: "Medium",
    2: "High"
}

humidity_sales.index = humidity_sales.index.map(humidity_map)

import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

humidity_sales.plot(kind="bar")

plt.title("Sales by Humidity Level")
plt.xlabel("Humidity")
plt.ylabel("Average Sales")

plt.show()

#iv)impact on weather severity
eda_sample = price_calendar_valid.sample(n=50000, random_state=42)

severity_sales = eda_sample.groupby("weather_severity")["sales"].mean()

severity_map = {
    0: "Good",
    1: "Moderate",
    2: "Bad"
}

severity_sales.index = severity_sales.index.map(severity_map)

import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

severity_sales.plot(kind="bar")

plt.title("Sales by Weather Severity")
plt.xlabel("Weather Condition")
plt.ylabel("Average Sales")

plt.show()

#v)event type vs sales
# take sample (avoid memory issue)
eda_sample = price_calendar_valid.sample(n=50000, random_state=42)

# groupby
event_sales = eda_sample.groupby("event_type")["sales"].mean()

import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

event_sales.plot(kind="bar")

plt.title("Average Sales by Event Type")
plt.xlabel("Event Type")
plt.ylabel("Average Sales")

plt.show()

#vi)event name vs sales
top_events = eda_sample.groupby("event_name")["sales"].mean().sort_values(ascending=False).head(10)

import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))

top_events.plot(kind="bar")

plt.title("Top Events by Sales")
plt.xlabel("Event Name")
plt.ylabel("Average Sales")

plt.xticks(rotation=45)
plt.show()


#vii)weekends vs weekdays impact on sales
weekend_sales = eda_sample.groupby("is_weekend")["sales"].mean()

weekend_map = {
    0: "Weekday",
    1: "Weekend"
}

weekend_sales.index = weekend_sales.index.map(weekend_map)

plt.figure(figsize=(5,4))

weekend_sales.plot(kind="bar")

plt.title("Sales: Weekday vs Weekend")
plt.xlabel("Day Type")
plt.ylabel("Average Sales")

plt.show()

#Sales Distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
price_calendar_valid["sales"].hist(bins=50)
plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

#sales trend over  time
import numpy as np

plt.figure(figsize=(8,5))
np.log1p(price_calendar_valid["sales"]).hist(bins=50)
plt.title("Log Transformed Sales Distribution")
plt.xlabel("Log(Sales)")
plt.ylabel("Frequency")
plt.show()

#zero sales percentage
zero_sales = (price_calendar_valid["sales"] == 0).mean() * 100
print("Zero Sales %:", zero_sales)

#Sales Trend Over Time
import matplotlib.pyplot as plt

# Convert date
price_calendar_valid["date"] = pd.to_datetime(price_calendar_valid["date"])

# Group by date (VERY IMPORTANT)
daily_sales = price_calendar_valid.groupby("date")["sales"].sum()

# Plot
plt.figure(figsize=(10,5))
daily_sales.plot()

plt.title("Daily Sales Trend Over Time")
plt.xlabel("Date")
plt.ylabel("Total Sales")

plt.show()


daily_sales.rolling(7).mean().plot(figsize=(10,5))
plt.title("7-Day Moving Average Sales")
plt.show()


###Model Training
#feature selection
# ================================
# 1. Feature Selection
# ================================
X = price_calendar_valid[
    [
        "sell_price",
        "lag_7",
        "rolling_mean_7",
        "month",
        "day_of_week",
        "event_type",
        "store_id",
        "cat_id"
    ]
].copy()

y = price_calendar_valid["sales"]

# ================================
# 2. Encoding
# ================================
from sklearn.preprocessing import LabelEncoder

le_event = LabelEncoder()
le_store = LabelEncoder()
le_cat = LabelEncoder()

X["event_type"] = le_event.fit_transform(X["event_type"])
X["store_id"] = le_store.fit_transform(X["store_id"])
X["cat_id"] = le_cat.fit_transform(X["cat_id"])

# ================================
# 3. Train-Test Split
# ================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ================================
# 4. Linear Regression Model
# ================================
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

# ================================
# ⭐ 5. COEFFICIENT EXPLANATION
# ================================
import pandas as pd

coeff_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

# SORT AFTER CREATION (important)
coeff_df = coeff_df.sort_values(by="Coefficient", ascending=False)

print("===== Linear Regression Coefficients =====")
print(coeff_df)

# ================================
# 6. Predictions
# ================================
y_pred = model.predict(X_test)

# ================================
# 7. Evaluation
# ================================
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("\n===== Linear Regression Performance =====")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# ================================
# 8. Ridge Regression
# ================================
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

print("\n===== Ridge Performance =====")
print("Ridge R2:", r2_score(y_test, y_pred_ridge))

# ================================
# 9. Plots
# ================================
import matplotlib.pyplot as plt

# Actual vs Predicted
import numpy as np
import matplotlib.pyplot as plt

# sort values for clean visualization
sorted_idx = np.argsort(y_test)

y_test_sorted = y_test.iloc[sorted_idx]
y_pred_sorted = y_pred[sorted_idx]

plt.figure(figsize=(10,5))
plt.plot(y_test_sorted.values, label="Actual", linewidth=2)
plt.plot(y_pred_sorted, label="Predicted", linewidth=2)

plt.title("Actual vs Predicted Sales (Sorted)")
plt.xlabel("Samples (sorted)")
plt.ylabel("Sales")
plt.legend()
plt.show()

# Error Distribution
errors = y_test - y_pred

plt.figure(figsize=(6,4))

plt.hist(errors, bins=50)

plt.xlim(-20, 20)   # 👈 VERY IMPORTANT (focus only useful range)

plt.title("Error Distribution (Zoomed)")
plt.xlabel("Error")
plt.ylabel("Frequency")
plt.show()

# Residual Plot
plt.figure(figsize=(6,4))
plt.scatter(y_pred, errors, alpha=0.3)
plt.axhline(y=0, color='r')
plt.xlabel("Predicted")
plt.ylabel("Error")
plt.title("Residual Plot")
plt.show()

# Random Forest
# ================================
# RANDOM FOREST (OPTIMIZED)
# ================================
# ================================
# 1. Imports
# ================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ================================
# 2. Create SAMPLE (IMPORTANT)
# ================================
data_sample = X.copy()
data_sample["sales"] = y

data_sample = data_sample.sample(n=100000, random_state=42)

X_sample = data_sample.drop("sales", axis=1)
y_sample = data_sample["sales"]

# ================================
# 3. Train-Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample,
    test_size=0.2,
    random_state=42
)

# ================================
# 4. Random Forest Model
# ================================
rf_model = RandomForestRegressor(
    n_estimators=20,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# ================================
# 🔥 5. SHAP (ADD HERE)
# ================================
import shap

# create explainer
rf_explainer = shap.TreeExplainer(rf_model)

# small sample for SHAP (very important)
X_shap_sample = X_test.sample(200, random_state=42)

# calculate shap values
rf_shap_values = rf_explainer.shap_values(X_shap_sample)

# 🔹 Summary Plot (main)
shap.summary_plot(rf_shap_values, X_shap_sample)

# 🔹 Bar Plot (simple importance)
shap.summary_plot(rf_shap_values, X_shap_sample, plot_type="bar")

# 🔹 Single prediction explanation (best for viva)
shap.plots.waterfall(
    shap.Explanation(
        values=rf_shap_values[0],
        base_values=rf_explainer.expected_value[0] if
    isinstance(rf_explainer.expected_value,(list,np.ndarray))
    else rf_explainer.expected_value, 
        data=X_shap_sample.iloc[0],
        feature_names=X_shap_sample.columns
    )
)

# ================================
# 6. Predictions
# ================================
y_pred_rf = rf_model.predict(X_test)

# ================================
# 7. Evaluation
# ================================
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("===== Random Forest Results =====")
print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

# ================================
# 8. Actual vs Predicted
# ================================
y_test_sample = y_test[:100]
y_pred_sample = y_pred_rf[:100]

plt.figure(figsize=(8,4))
plt.plot(y_test_sample.values, label="Actual")
plt.plot(y_pred_sample, label="Predicted")

plt.legend()
plt.title("Actual vs Predicted Sales")
plt.xlabel("Samples")
plt.ylabel("Sales")
plt.show()

# ================================
# 9. Sorted Comparison
# ================================
sorted_idx = np.argsort(y_test.values)

y_test_sorted = y_test.values[sorted_idx]
y_pred_sorted = y_pred_rf[sorted_idx]

plt.figure(figsize=(8,4))
plt.plot(y_test_sorted[:100], label="Actual")
plt.plot(y_pred_sorted[:100], label="Predicted")

plt.legend()
plt.title("Sorted Actual vs Predicted")
plt.xlabel("Samples (sorted)")
plt.ylabel("Sales")
plt.show()
#XGBOOST
# ================================
# XGBOOST REGRESSION MODEL
# ================================
# ================================
# 1. Train XGBoost Model
# ================================
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# ================================
# 2. Predictions
# ================================
y_pred_xgb = xgb_model.predict(X_test)

# ================================
# 3. Evaluation
# ================================
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("===== XGBoost Results =====")
print("MAE :", mae_xgb)
print("RMSE:", rmse_xgb)
print("R2 Score:", r2_xgb)

# ================================
# 4. SHAP (IMPORTANT PART)
# ================================
import shap

# 🔥 Use small sample (VERY IMPORTANT for speed)
X_shap_sample = X_train.sample(n=2000, random_state=42)

# Create explainer
explainer = shap.TreeExplainer(xgb_model)

# Compute SHAP values
shap_values = explainer.shap_values(X_shap_sample)

# ================================
# 5. SHAP Summary Plot (GLOBAL)
# ================================
shap.summary_plot(shap_values, X_shap_sample)

# ================================
# 6. SHAP Bar Plot (Feature Importance)
# ================================
shap.summary_plot(shap_values, X_shap_sample, plot_type="bar")

# ================================
# 7. SHAP Waterfall (Single Prediction)
# ================================
# 🔥 VERY IMPORTANT FIX (no error)
sample_index = 0

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_index],
        base_values=explainer.expected_value,
        data=X_shap_sample.iloc[sample_index],
        feature_names=X_shap_sample.columns
    )
)

##############power BI
final_df = price_calendar_valid[
    [
        "date",
        "sales",
        "sell_price",
        "store_id",
        "cat_id",
        "day_of_week",
        "month"
    ]
].copy()

final_df["revenue"] = final_df["sales"] * final_df["sell_price"]
final_df.to_csv("C:/Users/new/Downloads/sales_dashboard_data.csv",index=False)
print("saved to downloads")
final_df.head()

print(final_df['date'].nunique())
print(final_df['sell_price'].nunique())
print(final_df['sales'].nunique())

final_df.nunique()

# ==========================================
# # FINAL EXPORT FOR POWER BI (ALL FILES)
# # ==========================================

# # ==========================================
# # FINAL CLEAN EXPORT (ERROR-FREE VERSION)
# # ==========================================

import pandas as pd
import numpy as np
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================
# 1. USE SAME TEST SET FOR ALL MODELS
# ==========================================

# Use the SAME X_test, y_test for consistency
y_test_clean = np.array(y_test)

# Predictions from models
y_pred_lr = np.array(y_pred)          # Linear
y_pred_rf_clean = np.array(y_pred_rf) # Random Forest
y_pred_xgb_clean = np.array(y_pred_xgb) # XGBoost

# Ensure equal length (VERY IMPORTANT SAFETY)
min_len = min(len(y_test_clean), len(y_pred_lr), len(y_pred_rf_clean), len(y_pred_xgb_clean))

y_test_clean = y_test_clean[:min_len]
y_pred_lr = y_pred_lr[:min_len]
y_pred_rf_clean = y_pred_rf_clean[:min_len]
y_pred_xgb_clean = y_pred_xgb_clean[:min_len]

# ==========================================
# 2. PREDICTIONS FILE
# ==========================================

pred_df = pd.DataFrame({
    "actual_sales": y_test_clean,
    "linear_pred": y_pred_lr,
    "rf_pred": y_pred_rf_clean,
    "xgb_pred": y_pred_xgb_clean
})

pred_df.to_csv("C:/Users/new/Downloads/predictions.csv", index=False)

print("✔ predictions.csv created")

# ==========================================
# 3. METRICS (FOR BEST MODEL - XGBOOST)
# ==========================================

mae_xgb = mean_absolute_error(y_test_clean, y_pred_xgb_clean)
rmse_xgb = np.sqrt(mean_squared_error(y_test_clean, y_pred_xgb_clean))
r2_xgb = r2_score(y_test_clean, y_pred_xgb_clean)

metrics_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [mae_xgb, rmse_xgb, r2_xgb]
})

metrics_df.to_csv("C:/Users/new/Downloads/metrics.csv", index=False)

print("✔ metrics.csv created")

# # ==========================================
# # 4. MODEL COMPARISON (FIXED)
# # ==========================================

mae_lr = mean_absolute_error(y_test_clean, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test_clean, y_pred_lr))
r2_lr = r2_score(y_test_clean, y_pred_lr)

mae_rf = mean_absolute_error(y_test_clean, y_pred_rf_clean)
rmse_rf = np.sqrt(mean_squared_error(y_test_clean, y_pred_rf_clean))
r2_rf = r2_score(y_test_clean, y_pred_rf_clean)

model_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "MAE": [mae_lr, mae_rf, mae_xgb],
    "RMSE": [rmse_lr, rmse_rf, rmse_xgb],
    "R2": [r2_lr, r2_rf, r2_xgb]
})

model_comparison.to_csv("C:/Users/new/Downloads/model_comparison.csv", index=False)
print("✔ model_comparison.csv created")

# # ==========================================
# # 5. SHAP FEATURE IMPORTANCE (XGBOOST)
# # ==========================================

explainer = shap.TreeExplainer(xgb_model)

# small sample for speed
X_shap_sample = X_test.sample(n=1000, random_state=42)

shap_values = explainer.shap_values(X_shap_sample)

shap_importance = pd.DataFrame({
    "feature": X_shap_sample.columns,
    "importance": np.abs(shap_values).mean(axis=0)
}).sort_values(by="importance", ascending=False)

shap_importance.to_csv("C:/Users/new/Downloads/shap_importance.csv", index=False)

print("✔ shap_importance.csv created")

# ==========================================
# 6. FINAL CONFIRMATION
# ==========================================

print("\n✅ ALL FILES CREATED SUCCESSFULLY:")
print("1. predictions.csv")
print("2. metrics.csv")
print("3. model_comparison.csv")
print("4. shap_importance.csv")



# ==========================================
# FINAL ADVANCED DASHBOARD EXPORT
# ==========================================

dashboard_df = price_calendar_valid[
    [
        # Time Features
        "date",
        "day_of_week",
        "week_of_year",
        "month",
        "year",

        # Business Features
        "sales",
        "revenue",
        "sell_price",

        # Product Features
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",

        # Trend Features
        "lag_7",
        "lag_14",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_std_7",
        "sales_growth",
        "sales_momentum",

        # Price Features
        "price_change",

        # Weekend Feature
        "is_weekend",

        # Weather Features
        "temperature",
        "humidity",
        "wind_speed",
        "weather_type",
        "weather_severity",
        "temp_level",
        "humidity_level",
        "wind_level",

        # Event Features
        "event_name",
        "event_type",

        # Demand Features
        "demand_level",

        # Readable Names
        "store_name",
        "dept_name"
    ]
].copy()

# ==========================================
# OPTIONAL: CREATE CATEGORY NAMES
# ==========================================

dashboard_df["category_name"] = dashboard_df["cat_id"].map({
    0: "HOBBIES",
    1: "HOUSEHOLD",
    2: "FOODS"
})

# ==========================================
# SAVE FILE
# ==========================================

dashboard_df.to_csv(
    "C:/Users/new/Downloads/dashboard_data.csv",
    index=False
)

print("✅ dashboard_data.csv exported successfully")
print("Shape:", dashboard_df.shape)
print("Columns:", dashboard_df.columns.tolist())



dashboard_sample = dashboard_df.sample(
    n=500000,
    random_state=42
)

dashboard_sample.to_csv(
    "C:/Users/new/Downloads/dashboard_sample.csv",
    index=False
)

print("✅ dashboard_sample.csv created") 
\
import os
import pandas as pd

print(os.listdir(r"C:\Users\new\Downloads"))



dashboard_df.to_csv(
    r"C:\Users\new\Downloads\dashboard_data.csv",
    index=False
)

pred_df.to_csv(
    r"C:\Users\new\Downloads\predictions.csv",
    index=False
)

metrics_df.to_csv(
    r"C:\Users\new\Downloads\metrics.csv",
    index=False
)

model_comparison.to_csv(
    r"C:\Users\new\Downloads\model_comparison.csv",
    index=False
)

shap_importance.to_csv(
    r"C:\Users\new\Downloads\shap_importance.csv",
    index=False
)

print("✅ ALL FILES SAVED")
